# EDA — Indian Weather (T024)

Notebook de **análise exploratória** para o marco M02 / história S02. Restrição da equipa: **PyArrow** + **Matplotlib** + **NumPy** (sem pandas nem scikit-learn).

## Amostra e reprodutibilidade

A primeira célula de código define `N_AMOSTRA` (linhas lidas do Parquet após seleção de colunas). **Gráficos e estatísticas descritas aqui referem-se a essa amostra**, não necessariamente ao ficheiro completo, salvo indicação em contrário.

## Documentação relacionada

- Alvo e pipeline: [preprocessamento.md](../docs/preprocessamento.md)
- Desequilíbrio de classes: [imbalance.md](../docs/imbalance.md)


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Tamanho da amostra (linhas) após leitura com colunas selecionadas
N_AMOSTRA = 500_000


def resolve_repo_and_fig_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    roots = [cwd]
    if cwd.name == "notebooks":
        roots.append(cwd.parent)
    for root in roots:
        if (root / "requirements.txt").exists():
            fig = root / "notebooks" / "figuras"
            fig.mkdir(parents=True, exist_ok=True)
            return root, fig
    fig = cwd / "figuras"
    fig.mkdir(parents=True, exist_ok=True)
    return cwd, fig


REPO_ROOT, FIG_DIR = resolve_repo_and_fig_dir()
PARQUET = REPO_ROOT / "data" / "Indian_Weather_Dataset.parquet"

os.environ.setdefault("MPLBACKEND", "Agg")

COLS_LEITURA = [
    "datetime",
    "rain_label",
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
    "lat",
    "lon",
    "hour",
    "month",
]

if not PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {PARQUET}. Coloque o dataset em data/ e execute a partir da raiz do repositório ou abra o Jupyter com cwd na raiz."
    )

table = pq.read_table(PARQUET, columns=COLS_LEITURA, use_threads=True)
n_full = table.num_rows
if table.num_rows > N_AMOSTRA:
    table = table.slice(0, N_AMOSTRA)

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", FIG_DIR)
print("Linhas no ficheiro (aprox.):", n_full)
print("Linhas usadas neste notebook:", table.num_rows)
print(table.schema)


## Distribuição do alvo (`rain_label`)

Histograma de frequências absolutas. Para interpretação do desequilíbrio e estratégias (class weights, oversampling no Spark, etc.), ver [imbalance.md](../docs/imbalance.md).


In [ ]:
labels = table.column("rain_label")
vc = pc.value_counts(labels)
labs = vc.field(0).to_numpy(zero_copy_only=False)
cnts = vc.field(1).to_numpy(zero_copy_only=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(x) for x in labs], cnts, color="#4C72B0")
ax.set_xlabel("rain_label")
ax.set_ylabel("Contagem (amostra)")
ax.set_title("Distribuição do alvo na amostra")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_rain_label_distribuicao.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_rain_label_distribuicao.png")
for a, b in zip(labs, cnts):
    print(a, int(b))


## Relações feature–alvo

Médias de variáveis numéricas por classe (`rain_label`) via `Table.group_by` do PyArrow, e **boxplot** de `precip_mm` por classe (distribuição condicional na amostra).


In [ ]:
agg_cols = [
    ("temperature_C", "mean"),
    ("humidity_pct", "mean"),
    ("cloud_cover_pct", "mean"),
]
g = table.group_by("rain_label").aggregate(agg_cols)
print(g)

labs_mean = g.column("rain_label").to_pylist()
series = {col: g.column(f"{col}_mean").to_numpy(zero_copy_only=False) for col, _ in agg_cols}
x = np.arange(len(labs_mean))
w = 0.25
fig, ax = plt.subplots(figsize=(8, 4))
for i, (col, _) in enumerate(agg_cols):
    ax.bar(x + (i - 1) * w, series[col], width=w, label=col)
ax.set_xticks(x)
ax.set_xticklabels([str(v) for v in labs_mean])
ax.set_xlabel("rain_label")
ax.set_ylabel("Média na amostra")
ax.set_title("Médias de preditores por classe (amostra)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_medias_num_por_rain_label.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_medias_num_por_rain_label.png")


In [ ]:
y = table.column("rain_label").to_numpy(zero_copy_only=False)
p = table.column("precip_mm").to_numpy(zero_copy_only=False)
classes = np.unique(y[~np.isnan(y)])
data = []
labels_txt = []
for c in classes:
    m = y == c
    vals = p[m & ~np.isnan(p)]
    data.append(vals)
    labels_txt.append(str(int(c)) if float(c).is_integer() else str(c))

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(data, showfliers=False)
ax.set_xticks(np.arange(1, len(labels_txt) + 1))
ax.set_xticklabels(labels_txt)
ax.set_xlabel("rain_label")
ax.set_ylabel("precip_mm")
ax.set_title("precip_mm por classe — amostra, outliers omitidos no gráfico")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_precip_boxplot_por_rain_label.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_precip_boxplot_por_rain_label.png")


## Correlação entre preditores (amostra)

Subconjunto de colunas numéricas alinhadas a [preprocessamento.md](../docs/preprocessamento.md). A matriz usa `numpy.corrcoef`; valores ausentes são substituídos temporariamente pela média da coluna **só para este cálculo exploratório** (não substitui decisões do pipeline T022).


In [ ]:
corr_cols = [
    "temperature_C",
    "humidity_pct",
    "precip_mm",
    "cloud_cover_pct",
    "pressure_hPa",
    "dew_point_C",
    "wind_speed_ms",
    "solar_radiation_Wm2",
]
rows = []
for c in corr_cols:
    v = table.column(c).to_numpy(zero_copy_only=False).astype("float64", copy=False)
    rows.append(v)
X = np.vstack(rows)
col_means = np.nanmean(X, axis=1, keepdims=True)
X_filled = np.where(np.isnan(X), col_means, X)
C = np.corrcoef(X_filled)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Correlação de Pearson — amostra; NaN imputados pela média da coluna")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_correlacao_preditoras.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_correlacao_preditoras.png")


## Dimensão temporal (`hour`, `month`, `datetime`)

Exploração de sazonalidade horária e mensal e de contagem de registos por dia (data truncada a partir de `datetime`).


In [ ]:
hour = table.column("hour").to_numpy(zero_copy_only=False)
month = table.column("month").to_numpy(zero_copy_only=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(hour[~np.isnan(hour)], bins=np.arange(-0.5, 24, 1), color="#55A868")
ax1.set_xlabel("hour")
ax1.set_ylabel("Contagem")
ax1.set_title("Distribuição horária (amostra)")

ax2.hist(month[~np.isnan(month)], bins=np.arange(0.5, 13, 1), color="#C44E52")
ax2.set_xlabel("month")
ax2.set_ylabel("Contagem")
ax2.set_title("Distribuição mensal (amostra)")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_hour_month_histogramas.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_hour_month_histogramas.png")


In [ ]:
d_arr = pc.cast(table.column("datetime"), pa.date32())
st = pc.value_counts(d_arr)
days = st.field(0).to_numpy(zero_copy_only=False)
cnts = st.field(1).to_numpy(zero_copy_only=False)
order = np.argsort(days)
days_s = days[order]
cnts_s = cnts[order].astype(float)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(days_s, cnts_s, linewidth=0.8)
ax.set_xlabel("Data")
ax.set_ylabel("Nº de linhas (amostra)")
ax.set_title("Contagem por dia na amostra (value_counts sobre date)")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_contagem_por_dia.png", dpi=150)
plt.close(fig)
print("Figura:", FIG_DIR / "eda_contagem_por_dia.png")


## Figuras exportadas (caminhos)

Relativos à raiz do repositório:

- `notebooks/figuras/eda_rain_label_distribuicao.png`
- `notebooks/figuras/eda_medias_num_por_rain_label.png`
- `notebooks/figuras/eda_precip_boxplot_por_rain_label.png`
- `notebooks/figuras/eda_correlacao_preditoras.png`
- `notebooks/figuras/eda_hour_month_histogramas.png`
- `notebooks/figuras/eda_contagem_por_dia.png`

## Insights acionáveis (Parte 3)

1. **Alvo:** validar na amostra (e depois no Parquet completo com `profile_parquet` / Spark) se a distribuição de `rain_label` exige **class weights** ou **rebalanceamento** conforme [imbalance.md](../docs/imbalance.md); alinhar métricas (F1, PR-AUC) ao desequilíbrio.
2. **Chuva (`precip_mm`):** caudas pesadas por classe sugerem que modelos baseados em **árvores** ou transformações robustas podem ser mais estáveis do que assumir Gaussianidade; manter imputação/mediana **fit só em treino** (T022).
3. **Correlações fortes** (ex.: termodinâmica entre temperatura, humidade, ponto de orvalho): considerar **redundância** e custo de features em Spark; árvores lidam bem, modelos lineares podem precisar de regularização ou seleção.
4. **Temporal:** picos em certas horas/meses reforçam utilidade de `hour` / `month` (ou representação cíclica sin/cos) já prevista no pré-processamento; **split temporal** (T021) evita vazamento quando se exploram tendências por dia.
5. **Próximo passo:** materializar splits e pipeline em Spark com **fit apenas no treino** e validar generalização na janela temporal de validação/teste.
